In [1]:

import os
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
)

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)

from openai import OpenAI

DATA_PATH = "datset.csv"   
OPENAI_MODEL = "gpt-5-mini"   # lower-cost LLM for feature engineering
DISTILBERT_MODEL = "distilbert-base-uncased"
MAX_LEN = 256
LOOKBACK_TXNS = 10
USE_LLM = True
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
import os


# client = OpenAI()


/data1/rachit/.conda/envs/cond/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
DATA_PATH="../../transactions/card_transaction.v1.csv"
df = pd.read_csv(DATA_PATH)

In [4]:
print(df.dtypes)
print(df.head())

User                int64
Card                int64
Year                int64
Month               int64
Day                 int64
Time               object
Amount             object
Use Chip           object
Merchant Name       int64
Merchant City      object
Merchant State     object
Zip               float64
MCC                 int64
Errors?            object
Is Fraud?          object
dtype: object
   User  Card  Year  Month  Day   Time   Amount           Use Chip  \
0     0     0  2002      9    1  06:21  $134.09  Swipe Transaction   
1     0     0  2002      9    1  06:42   $38.48  Swipe Transaction   
2     0     0  2002      9    2  06:22  $120.34  Swipe Transaction   
3     0     0  2002      9    2  17:45  $128.95  Swipe Transaction   
4     0     0  2002      9    3  06:23  $104.71  Swipe Transaction   

         Merchant Name  Merchant City Merchant State      Zip   MCC Errors?  \
0  3527213246127876953       La Verne             CA  91750.0  5300     NaN   
1  -7276120921399

In [5]:
for col in df.columns:
    print(col, ":", df[col].nunique())
    
for col in df.columns:
    print(f"\nColumn: {col}")
    print(df[col].unique())

User : 2000
Card : 9
Year : 30
Month : 12
Day : 31
Time : 1440
Amount : 98953
Use Chip : 3
Merchant Name : 100343
Merchant City : 13429
Merchant State : 223
Zip : 27321
MCC : 109
Errors? : 23
Is Fraud? : 2

Column: User
[   0    1    2 ... 1997 1998 1999]

Column: Card
[0 1 2 3 4 5 6 7 8]

Column: Year
[2002 2003 2004 2005 2006 2007 2008 2009 2010 2011 2012 2013 2014 2015
 2016 2017 2018 2019 2020 1999 2000 2001 1998 1996 1997 1995 1994 1991
 1992 1993]

Column: Month
[ 9 10 11 12  1  2  3  4  5  6  7  8]

Column: Day
[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31]

Column: Time
['06:21' '06:42' '06:22' ... '01:48' '00:33' '00:42']

Column: Amount
['$134.09' '$38.48' '$120.34' ... '$588.73' '$1201.00' '$584.63']

Column: Use Chip
['Swipe Transaction' 'Online Transaction' 'Chip Transaction']

Column: Merchant Name
[ 3527213246127876953  -727612092139916043  3414527459579106770 ...
 -4348891722741102135  -642409450154660123 -353358046456151

In [38]:

df["Amount"] = df["Amount"].replace('[\$,]', '', regex=True).astype(float)

df["Hour"] = pd.to_datetime(df["Time"], errors="coerce").dt.hour

df["Use Chip"] = df["Use Chip"].map({"Yes": 1, "No": 0}).fillna(0)

df["Errors?"] = (df["Errors?"] != "None").astype(int)

df["Zip"] = pd.to_numeric(df["Zip"], errors="coerce")

df["Time"] = pd.to_numeric(df["Time"], errors="coerce")

df["Hour"] = (df["Time"] // 3600) % 24
df["Is Fraud?"] = df["Is Fraud?"].map({"Yes": 1, "No": 0})

df = df.fillna(0)

# print(df.dtypes)
# print(df.head())

/tmp/ipykernel_806624/3282332532.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Hour"] = pd.to_datetime(df["Time"], errors="coerce").dt.hour


In [39]:
df["Merchant City"] = df["Merchant City"].astype(str)
df["Merchant State"] = df["Merchant State"].astype(str)

In [40]:
print(df.dtypes)
print(df.head())

User                int64
Card                int64
Year                int64
Month               int64
Day                 int64
Time              float64
Amount            float64
Use Chip          float64
Merchant Name       int64
Merchant City      object
Merchant State     object
Zip               float64
MCC                 int64
Errors?             int64
Is Fraud?           int64
Hour              float64
dtype: object
   User  Card  Year  Month  Day  Time  Amount  Use Chip        Merchant Name  \
0     0     0  2002      9    1   0.0  134.09       0.0  3527213246127876953   
1     0     0  2002      9    1   0.0   38.48       0.0  -727612092139916043   
2     0     0  2002      9    2   0.0  120.34       0.0  -727612092139916043   
3     0     0  2002      9    2   0.0  128.95       0.0  3414527459579106770   
4     0     0  2002      9    3   0.0  104.71       0.0  5817218446178736267   

   Merchant City Merchant State      Zip   MCC  Errors?  Is Fraud?  Hour  
0       La Ver

## Run form here

In [7]:
# STEP 4: Connect to Ollama
import requests

def call_llm(prompt):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3",
            "prompt": prompt,
            "stream": False
        }
    )

    data = response.json()
    
    if "response" in data:
        return data["response"]
    else:
        print("Error:", data)
        return None

In [ ]:
Prompt = """
==============================
DATA SUMMARY
==============================

- Number of users: 2000
- Transactions span: multiple years (1991–2020)
- Amount has high variability (~100K unique values)
- Merchant Name is high-cardinality (~100K unique)
- Merchant City is high-cardinality (~13K)
- Merchant State includes global locations (223 unique)
- Errors column contains multiple error types (including combinations)
- Fraud label is binary (Yes/No)

==============================
VALUE DISTRIBUTION INSIGHTS
==============================

Use Chip:
- Swipe Transaction
- Online Transaction
- Chip Transaction

Errors?:
- NaN (no error)
- Technical Glitch
- Insufficient Balance
- Bad PIN
- Bad CVV
- Bad Zipcode
- Combinations of above errors

Merchant State:
- Mix of US states and international locations

Is Fraud:
- Yes / No

==============================
SAMPLE DATA (STRUCTURE)
==============================

Each row represents a transaction:

User=0, Card=0, Year=2002, Month=9, Day=1, Time=06:21,
Amount=$134.09, Use Chip=Swipe Transaction,
Merchant Name=high-cardinality ID,
Merchant City=La Verne, Merchant State=CA,
Zip=91750, MCC=5300, Errors?=NaN, Is Fraud=No

==============================
PREPROCESSING & MISSING VALUE HANDLING
==============================

Before feature computation, assume the following preprocessing has been applied:

1. Amount:
   - Remove "$" and convert to float
   - Missing values (if any) → fill with median Amount

2. Time:
   - Convert "HH:MM" → Hour (0–23)
   - Missing values → fill with mode Hour

3. Use Chip:
   - Encode as:
       Swipe Transaction → 0
       Chip Transaction → 1
       Online Transaction → 2
   - Missing values → fill with most frequent category

4. Errors?:
   - NaN → 0 (no error)
   - Any error string → 1 (error present)

5. Merchant City / Merchant State:
   - Missing values → fill with "Unknown"

6. Zip:
   - Missing values → fill with median Zip

7. MCC:
   - Missing values → fill with mode MCC

8. Is Fraud:
   - Yes → 1, No → 0
   - MUST NOT be used for feature generation

After preprocessing, the following cleaned columns are available:

- Amount (float)
- Hour (int)
- Use Chip (int)
- Errors? (int)
- MCC (int)
- Merchant City (string)
- Merchant State (string)
- Zip (float)


==============================
IMPORTANT DATA CHARACTERISTICS
==============================

- Merchant Name is extremely high-cardinality → avoid direct usage
- Merchant City/State have high diversity → use aggregation (nunique)
- Time is granular → useful for behavioral patterns
- Errors column contains multiple combined error types
- Amount has high variance → useful for anomaly detection



============================== 
YOUR TASK 
============================== 
Design USER-LEVEL features for fraud detection. Each feature MUST produce ONE value per User.

==============================
OUTPUT FORMAT (STRICT JSON)
==============================

Return a JSON list:

[
  {
    "feature_name": "...",
    "pandas_code": "...",
    "description": "...",
    "intuition": "...",
    "type": "numeric"
  }
]

- pandas_code must be executable
- must return a Series indexed by User
- no explanations outside JSON

==============================
EXECUTION REQUIREMENTS
==============================

- Features must be computed using pandas
- Final output must be a Series indexed by User
- Length must equal number of unique users
- Must not use undefined variables
- Must not use target column (Is Fraud)

==============================
ALLOWED OPERATIONS
==============================

You may use:

- multiple aggregations
- ratios (e.g., sum / count)
- differences (max - mean)
- standard deviation and variance
- conditional aggregations
- arithmetic combinations of features

==============================
QUALITY CONSTRAINTS
==============================

- Avoid redundant or highly correlated features
- Prefer interpretable features
- Prefer features that capture behavioral differences
- Avoid trivial features (e.g., constant or near-constant)

==============================
FINAL INSTRUCTIONS
==============================

- Generate 12–15 features
- Each feature must be unique and meaningful
- Output ONLY JSON
- Do NOT include explanations outside JSON

Ensure final output remains a valid Series per User.
"""

In [9]:

# from dotenv import load_dotenv
# import os
# import google.generativeai as genai

# load_dotenv()

# genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

In [10]:
# for m in genai.list_models():
#     print(m.name, "->", m.supported_generation_methods)

In [11]:
# model = genai.GenerativeModel("models/gemini-flash-lite-latest")

In [12]:

# df["Amount"] = df["Amount"].replace('[\$,]', '', regex=True).astype(float)

# df["Hour"] = pd.to_datetime(df["Time"], errors="coerce").dt.hour

# df["Use Chip"] = df["Use Chip"].map({"Yes": 1, "No": 0}).fillna(0)

# df["Errors?"] = (df["Errors?"] != "None").astype(int)

# df["Zip"] = pd.to_numeric(df["Zip"], errors="coerce")

# df = df.fillna(0)

In [13]:
# PROMPT = """
# You are a fraud analytics feature engineering assistant.

# The features you generate will be used to train a RandomForestClassifier.

# Dataset columns and TYPES (STRICT — DO NOT ASSUME ANYTHING ELSE):

# - User → int64 (group key)
# - Card → int64 
# - Year → int64
# - Month → int64
# - Day → int64
# - Time → float64 (transaction timestamp in seconds —  directly)
# - Hour → float64 (0–23, derived from Time — USE THIS)
# - Amount → float64 (numeric)
# - Use Chip → float64 (1 = Yes, 0 = No)
# - Merchant Name → int64 (encoded —  directly)
# - Merchant City → object (categorical — ONLY use nunique/count)
# - Merchant State → object (categorical — ONLY use nunique/count)
# - Zip → float64 (numeric)
# - MCC → int64 (category code — can use nunique/count)
# - Errors? → int64 (1 = error, 0 = no error)
# - Is Fraud? → int64 (target label — DO NOT USE)

# YOUR TASK:

# Design GLOBAL USER-LEVEL FEATURES and behavioral patterns that can help a Classifier detect fraud.

# STRICT REQUIREMENTS (VERY IMPORTANT):

# - Features must be NUMERIC only (int or float)
# - Must be directly usable in sklearn (no preprocessing needed)
# - Must use pandas groupby('User')
# - MUST be computed ONLY from df (no dependency on other generated features)
# - SAME features for ALL users


# AVOID:

# - raw text usage
# - using Merchant Name directly
# - using Is Fraud?
# - string operations
# - referencing other generated features

# IMPORTANT RULES:

# - Always use "Hour" for time-based features (NOT Time)
# - Merchant City / State → ONLY nunique or count
# - Ensure all outputs are numeric scalars per user

# OUTPUT FORMAT (STRICT JSON):

# [
#   {
#     "feature_name": "...",
#     "pandas_code": "...",
#     "description": "...",
#     "type": "numeric"
#   }
# ]

# Generate 12–15 high-quality fraud detection features.

# Focus on:
# - spending patterns
# - transaction frequency
# - merchant/category diversity
# - geographic spread
# - time behavior
# - error behavior
# - chip usage patterns

# Do NOT compute values.
# Do NOT output anything outside JSON.


# """

In [ ]:
# PROMPT = """
# You are a fraud analytics feature engineering assistant.

# The features you generate will be used to train a RandomForestClassifier.

# ==============================
# DATASET SCHEMA (STRICT)
# ==============================

# - User → int64 (group key)
# - Card → int64 
# - Year → int64
# - Month → int64
# - Day → int64
# - Time → float64 (transaction timestamp in seconds — DO NOT USE DIRECTLY)
# - Hour → float64 (0–23, derived from Time — USE THIS)
# - Amount → float64 (numeric)
# - Use Chip → float64 (1 = Yes, 0 = No)
# - Merchant Name → int64 (encoded — DO NOT USE DIRECTLY)
# - Merchant City → object (categorical — ONLY use nunique/count)
# - Merchant State → object (categorical — ONLY use nunique/count)
# - Zip → float64 (numeric)
# - MCC → int64 (category code — can use nunique/count)
# - Errors? → int64 (1 = error, 0 = no error)
# - Is Fraud? → int64 (target label — DO NOT USE)

# ==============================
# YOUR TASK
# ==============================

# Design USER-LEVEL features for fraud detection.

# Each feature MUST produce ONE value per User.

# ==============================
# STRICT REQUIREMENTS (VERY IMPORTANT)
# ==============================

# - Features must be NUMERIC only (int or float)
# - Must be directly usable in sklearn (no preprocessing needed)
# - MUST be computed ONLY from df
# - MUST NOT depend on other generated features
# - SAME features for ALL users

# FEATURE DESIGN FOCUS:

# - spending patterns
# - transaction frequency
# - merchant/category diversity
# - geographic spread
# - time behavior (USE Hour only)
# - error behavior
# - chip usage patterns

# ==============================
# CRITICAL RULE (VERY IMPORTANT)
# ==============================

# - Each feature must apply ONLY ONE aggregation
# - After df.groupby('User'), apply EXACTLY ONE aggregation
# - DO NOT apply aggregation on top of aggregation

# INVALID:
# df.groupby('User')['Amount'].mean().mean()
# df.groupby('User')['MCC'].nunique().mean()
# df.groupby('User').size().mean()

# VALID:
# df.groupby('User')['Amount'].mean()
# df.groupby('User')['MCC'].nunique()
# df.groupby('User').size()

# If the result is a single number (scalar), the feature is INVALID.

# ==============================
# CRITICAL EXECUTION RULES (MANDATORY)
# ==============================

# Each feature MUST follow EXACTLY this pattern:

# df.groupby('User')['COLUMN'].AGG()

# OR:

# df.groupby('User').size()

# Where:
# - COLUMN = exactly ONE column
# - AGG MUST be one of: mean, sum, count, nunique, max, min, std

# ==============================
# ABSOLUTE RESTRICTIONS (NO EXCEPTIONS)
# ==============================

# - NEVER use df['User'].map()
# - NEVER use .values
# - NEVER use reset_index()
# - NEVER use multiple columns (NO [['col1','col2']])
# - NEVER use resample()
# - NEVER use datetime operations
# - NEVER return DataFrame (must return Series)
# - NEVER perform arithmetic between multiple aggregations
# - NEVER nest groupby
# - NEVER use apply()
# - NEVER chain column selection like: df.groupby(... )['col1']['col2']

# ==============================
# VALID EXAMPLES (ONLY THESE TYPES)
# ==============================

# df.groupby('User')['Amount'].mean()

# df.groupby('User')['MCC'].nunique()

# df.groupby('User')['Errors?'].sum()

# df.groupby('User')['Hour'].std()

# df.groupby('User').size()

# ==============================
# INVALID EXAMPLES (DO NOT GENERATE)
# ==============================

# df['User'].map(...)

# df.groupby('User')[['Amount','Hour']]

# df.groupby('User').resample('H')

# df.groupby('User')['Amount'].mean().reset_index()

# df.groupby('User')['Amount'].mean() / df.groupby('User').size()

# ==============================
# FEATURE DESIGN FOCUS
# ==============================

# - spending patterns (mean, std, max)
# - transaction frequency (counts)
# - merchant/category diversity (nunique)
# - geographic spread (city/state/zip diversity)
# - time behavior (Hour-based stats)
# - error behavior
# - chip usage patterns

# ==============================
# OUTPUT FORMAT (STRICT JSON)
# ==============================

# [
#   {
#     "feature_name": "...",
#     "pandas_code": "...",
#     "description": "...",
#     "type": "numeric"
#   }
# ]

# ==============================
# FINAL INSTRUCTIONS
# ==============================

# - Generate 12–15 high-quality features
# - Each feature MUST return a pandas Series indexed by User
# - Length MUST equal number of unique users
# - Do NOT compute values
# - Do NOT explain anything
# - Output ONLY JSON
# - Output MUST start with '[' and end with ']'
# - Use double quotes only
# """

In [15]:
# PROMPT="""You are a fraud analytics feature engineering assistant.

# The features you generate will be used to train a RandomForestClassifier.

# Dataset columns and TYPES (STRICT — DO NOT ASSUME ANYTHING ELSE):

# - User → int64 (group key)
# - Card → int64 
# - Year → int64
# - Month → int64
# - Day → int64
# - Time → float64 (transaction timestamp in seconds — DO NOT USE DIRECTLY)
# - Hour → float64 (0–23, derived from Time — USE THIS)
# - Amount → float64 (numeric)
# - Use Chip → float64 (1 = Yes, 0 = No)
# - Merchant Name → int64 (encoded — DO NOT USE DIRECTLY)
# - Merchant City → object (categorical — ONLY use nunique/count)
# - Merchant State → object (categorical — ONLY use nunique/count)
# - Zip → float64 (numeric)
# - MCC → int64 (category code — can use nunique/count)
# - Errors? → int64 (1 = error, 0 = no error)
# - Is Fraud? → int64 (target label — DO NOT USE)

# ---

# YOUR TASK:

# Design GLOBAL USER-LEVEL FEATURES and behavioral patterns that can help a RandomForestClassifier detect fraud.

# ---

# STRICT REQUIREMENTS (VERY IMPORTANT):

# - Features must be NUMERIC only (int or float)
# - Must be directly usable in sklearn (no preprocessing needed)
# - Must use pandas groupby('User')
# - MUST be computed ONLY from df (no dependency on other generated features)
# - SAME features for ALL users

# ---

# FEATURE DESIGN FOCUS:

# - spending patterns
# - transaction frequency
# - merchant/category diversity
# - geographic spread
# - time behavior (USE Hour only)
# - error behavior
# - chip usage patterns

# ---

# AVOID:

# - raw text usage
# - using Merchant Name directly
# - using Is Fraud?
# - string operations
# - referencing other generated features
# - using Time column
# - non-numeric outputs

# ---

# ==============================
# STRICT EXECUTION TEMPLATE (MANDATORY)
# ==============================

# Every feature MUST follow EXACTLY ONE of these two forms:

# FORM 1:
# df['User'].map(df.groupby('User')['COLUMN'].AGG())

# FORM 2:
# df['User'].map(df.groupby('User').size())

# Where:
# - COLUMN = exactly ONE column name (string)
# - AGG MUST be one of:
#   mean, sum, count, nunique, max, min, std

# ==============================
# ABSOLUTE RESTRICTIONS (NO EXCEPTIONS)
# ==============================

# - NEVER use .values
# - NEVER use reset_index()
# - NEVER use multiple columns (NO [['col1','col2']])
# - NEVER use resample()
# - NEVER use datetime operations
# - NEVER return DataFrame
# - NEVER perform arithmetic between two map() calls
# - NEVER nest groupby operations
# - NEVER use apply()
# - NEVER chain column selection like:
#   df.groupby(... )['col1']['col2']

# - You may ONLY select ONE column after groupby:
#   df.groupby('User')['COLUMN']

# ==============================
# VALID EXAMPLES (ONLY THESE TYPES)
# ==============================

# df['User'].map(df.groupby('User')['Amount'].mean())

# df['User'].map(df.groupby('User')['MCC'].nunique())

# df['User'].map(df.groupby('User')['Errors?'].sum())

# df['User'].map(df.groupby('User').size())

# ==============================
# INVALID EXAMPLES (DO NOT GENERATE)
# ==============================

# df.groupby('User').size().values

# df.groupby('User')[['Amount','Hour']]

# df.groupby('User').resample('H')

# df['User'].map(A) / df['User'].map(B)

# df.groupby('User')['Amount'].mean().reset_index()



# EXAMPLE (FOLLOW THIS EXACT STYLE):

# {
#   "feature_name": "avg_amount_per_user",
#   "pandas_code": "df['User'].map(df.groupby('User')['Amount'].mean())",
#   "description": "Average transaction amount per user",
#   "type": "numeric"
# }

# ---

# OUTPUT FORMAT (STRICT JSON):

# [
#   {
#     "feature_name": "...",
#     "pandas_code": "...",
#     "description": "...",
#     "type": "numeric"
#   }
# ]

# ---

# FINAL INSTRUCTIONS:

# - Generate 12–15 high-quality fraud detection features
# - Do NOT compute values
# - Do NOT explain anything
# - Do NOT output anything outside JSON
# - Ensure VALID JSON (double quotes only)
# ==============================
# FINAL REQUIREMENT
# ==============================

# If a feature cannot be written EXACTLY in one of the VALID forms above,
# DO NOT include it.
# """

In [16]:
# model_llm = genai.GenerativeModel("models/gemini-flash-lite-latest")

In [28]:
response = call_llm(PROMPT)
print("RAW OUTPUT:\n", response)

RAW OUTPUT:
 Here are the 12 features I came up with, following the strict guidelines:

[
  {
    "feature_name": "avg_hourly_spend",
    "pandas_code": "df.groupby('User')['Amount'].mean()",
    "description": "Average hourly spend per user",
    "type": "numeric"
  },
  {
    "feature_name": "num_transactions_per_day",
    "pandas_code": "df.groupby('User').size().max()",
    "description": "Number of transactions per day for each user",
    "type": "numeric"
  },
  {
    "feature_name": "merchant_category_diversity",
    "pandas_code": "df.groupby('User')['MCC'].nunique()",
    "description": "Unique merchant categories per user",
    "type": "numeric"
  },
  {
    "feature_name": "time_of_day_frequency",
    "pandas_code": "df.groupby('User')['Hour'].nunique()",
    "description": "Frequency of transactions at different times of day for each user",
    "type": "numeric"
  },
  {
    "feature_name": "chip_usage_rate",
    "pandas_code": "df.groupby('User')['Use Chip'].mean()",
    "

In [29]:
import re

def fix_code(code):
    # Fix missing quotes
    replacements = {
        "[Amount]": "['Amount']",
        "[Time]": "['Time']",
        "[Is Fraud?]": "['Is Fraud?']"
    }
    for k, v in replacements.items():
        code = code.replace(k, v)

    # 🔥 Fix tuple column selection → list
    code = re.sub(
        r"\[['\"](\w+)['\"],\s*['\"](\w+)['\"]\]",
        r"[[ '\1', '\2' ]]",
        code
    )

    # Also fix ('A','B') → ['A','B']
    code = re.sub(
        r"\(\s*'(\w+)'\s*,\s*'(\w+)'\s*\)",
        r"['\1', '\2']",
        code
    )

    return code

In [30]:
def extract_json(llm_output):
    try:
        json_str = re.search(r'\[.*\]', llm_output, re.DOTALL).group(0)
        return json.loads(json_str)
    except:
        print("Failed to parse JSON")
        print(llm_output)
        return []
    


In [31]:
# response = model_llm.generate_content(PROMPT)
response=fix_code(response)
features = extract_json(response)



user_df = pd.DataFrame()

user_df = pd.DataFrame(index=df['User'].unique())

for f in features:
    try:
        print("Applying:", f["feature_name"])
        
        result = eval(f["pandas_code"])
        
        # ---- VALIDATION ----
        if not isinstance(result, pd.Series):
            raise ValueError("Not a pandas Series")
        
        # ---- FORCE ALIGNMENT ----
        result = result.reindex(user_df.index)
        
        user_df[f["feature_name"]] = result
        
    except Exception as e:
        print("Error in", f["feature_name"], ":", e)
        

user_df["is_fraud_user"] = df.groupby("User")["Is Fraud?"].max()
user_df["is_fraud_user"] = user_df["is_fraud_user"].reindex(user_df.index)

user_df = user_df.fillna(0)

Applying: avg_hourly_spend
Error in avg_hourly_spend : agg function failed [how->mean,dtype->object]
Applying: num_transactions_per_day
Error in num_transactions_per_day : Not a pandas Series
Applying: merchant_category_diversity
Applying: time_of_day_frequency
Error in time_of_day_frequency : 'Column not found: Hour'
Applying: chip_usage_rate
Error in chip_usage_rate : agg function failed [how->mean,dtype->object]
Applying: average_transaction_amount
Error in average_transaction_amount : agg function failed [how->mean,dtype->object]
Applying: num_errors_per_user
Applying: time_of_day_spread
Error in time_of_day_spread : 'Column not found: Hour'
Applying: merchant_city_diversity
Applying: zip_diversity
Applying: hourly_transaction_frequency
Error in hourly_transaction_frequency : 'Column not found: Hour'


In [21]:
print("\nFinal Shape:", user_df.shape)
print(user_df.head())


Final Shape: (2000, 1)
  is_fraud_user
0           Yes
1           Yes
2           Yes
3           Yes
4           Yes


In [22]:
user_df["is_fraud_user"] = df.groupby("User")["Is Fraud?"].max()

In [23]:
print(user_df.shape)

(2000, 1)


In [24]:
# Separate label first
y = user_df["is_fraud_user"]

# Remove non-numeric features
X = user_df.drop(columns=["is_fraud_user"])
X = X.select_dtypes(include=["number"])

In [25]:
print(y.isna().sum())

0


In [26]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   
)

In [27]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    class_weight="balanced"   
)

model.fit(X_train, y_train)

ValueError: at least one array or dtype is required

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.88      0.71      0.78       131
           1       0.87      0.95      0.91       269

    accuracy                           0.87       400
   macro avg       0.87      0.83      0.85       400
weighted avg       0.87      0.87      0.87       400

ROC-AUC: 0.9080564147677289


In [ ]:
import pandas as pd

importance = pd.Series(model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print(importance)

Hourly Transaction Count        0.244449
Chip Usage Count                0.241336
MCC Diversity                   0.157656
Zip Code Count                  0.132716
City/State Zip Diversity        0.126288
Mean Hourly Spending            0.057324
Hourly Merchant Category Max    0.040232
Hourly Error Rate               0.000000
dtype: float64


In [ ]:
feature_names = [f["feature_name"] for f in features]

In [ ]:
top_features = importance.to_dict()

In [ ]:
report1 = classification_report(y_test, y_pred)
roc1 = roc_auc_score(y_test, y_prob)

llm_input = {
    "existing_features": feature_names,
    "classification_report": report1,
    "roc_auc": roc1,
    "top_features": top_features
}


In [ ]:
print(report1)
print(importance)
print(roc1)

              precision    recall  f1-score   support

           0       0.88      0.71      0.78       131
           1       0.87      0.95      0.91       269

    accuracy                           0.87       400
   macro avg       0.87      0.83      0.85       400
weighted avg       0.87      0.87      0.87       400

Hourly Transaction Count        0.244449
Chip Usage Count                0.241336
MCC Diversity                   0.157656
Zip Code Count                  0.132716
City/State Zip Diversity        0.126288
Mean Hourly Spending            0.057324
Hourly Merchant Category Max    0.040232
Hourly Error Rate               0.000000
dtype: float64
0.9080564147677289


In [ ]:
def get_prompt(llm_input):
  PROMPT = f"""
  You are a fraud analytics feature engineering expert.

  A RandomForestClassifier was trained on USER-LEVEL features.

  ==============================
  DATASET SCHEMA (STRICT)
  ==============================

  - User → int64 (group key)
  - Card → int64 (DO NOT USE)
  - Year → int64
  - Month → int64
  - Day → int64
  - Time → float64 (timestamp in seconds — DO NOT USE directly)
  - Hour → float64 (0–23 — USE THIS for time features)
  - Amount → float64
  - Use Chip → float64 (1 = Yes, 0 = No)
  - Merchant Name → int64 (DO NOT USE directly)
  - Merchant City → object (ONLY use nunique/count)
  - Merchant State → object (ONLY use nunique/count)
  - Zip → float64
  - MCC → int64
  - Errors? → int64 (1 = error, 0 = no error)
  - Is Fraud? → int64 (TARGET — DO NOT USE)

  ==============================
  CURRENT FEATURES
  ==============================
  {llm_input["existing_features"]}

  ==============================
  MODEL PERFORMANCE
  ==============================

  Classification Report:
  {llm_input["classification_report"]}

  ROC-AUC:
  {llm_input["roc_auc"]}

  Top Important Features:
  {llm_input["top_features"]}

  ==============================
  YOUR TASK
  ==============================

  1. Analyze weaknesses in the model (especially fraud recall)
  2. Identify missing behavioral signals
  3. Suggest 8–12 NEW or IMPROVED features

  ==============================
  STRICT RULES (VERY IMPORTANT)
  ==============================

  - Features must be NUMERIC only
  - Must work directly in sklearn (no preprocessing)
  - Must use pandas groupby('User')
  - MUST be directly computable from df
  - MUST NOT depend on other generated features
  - MUST NOT duplicate existing features


  AVOID:
  - raw text usage
  - using Merchant Name directly
  - using Is Fraud?
  - referencing other generated features
  - using Time (use Hour only)

  IMPORTANT:
  - Each pandas_code MUST return a pandas Series indexed by User

  ==============================
  OUTPUT FORMAT (STRICT JSON)
  ==============================

  [
    {{
      "feature_name": "...",
      "pandas_code": "...",
      "reason": "..."
    }}
  ]

  Focus on:
  - behavioral anomalies
  - spending variability
  - temporal patterns (Hour-based)
  - transaction bursts
  - merchant/category diversity
  - geographic spread
  - error behavior
  - chip usage changes

  Do NOT compute values.
  Do NOT output anything outside JSON.
  """
  return PROMPT

In [ ]:
# model_llm = genai.GenerativeModel("models/gemini-flash-lite-latest")
# response = model_llm.generate_content(PROMPT)


In [ ]:
# a=1
# for a in range(1,11):
#     PROMPT=get_prompt(llm_input)
#     response = call_llm(PROMPT)

#     output = response.text.replace("```json", "").replace("```", "").strip()

#     new_features = json.loads(output)
#     base_features = []

#     for f in features:
#         code = f["pandas_code"]
        
#         # Only keep df-based features
#         if "df.groupby" in code and "user_" not in code.split("=")[-1]:
#             base_features.append(f)
            
#     for f in base_features:
#         try:
#             user_df[f["feature_name"]] = eval(f["pandas_code"])
#         except Exception as e:
#             print("Base error:", f["feature_name"], e)
    
            
#     a+=1

In [ ]:
base_features = []

for f in features:
    code = f["pandas_code"]
    
    # Only keep df-based features
    if "df.groupby" in code and "user_" not in code.split("=")[-1]:
        base_features.append(f)

In [ ]:
for f in base_features:
    try:
        user_df[f["feature_name"]] = eval(f["pandas_code"])
    except Exception as e:
        print("Base error:", f["feature_name"], e)

Base error: Mean Hourly Amount (No Chip) Column(s) Amount already selected
Base error: Hourly Transaction Count (No Chip) Column(s) Time already selected
Base error: MCC Diversity (No Chip) Column(s) MCC already selected


In [ ]:
X = user_df.drop(columns=["is_fraud_user"])
y = user_df["is_fraud_user"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [ ]:

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)


y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# ================================
# 🔹 11. EVALUATE
# ================================
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred))

print("\n=== ROC AUC ===")
print(roc_auc_score(y_test, y_prob))

# ================================
# 🔹 12. FEATURE IMPORTANCE
# ================================
importance = pd.Series(model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print("\n=== Top Features ===")
print(importance.head(10))


=== Classification Report ===
              precision    recall  f1-score   support

           0       0.87      0.69      0.77       131
           1       0.86      0.95      0.90       269

    accuracy                           0.86       400
   macro avg       0.87      0.82      0.84       400
weighted avg       0.87      0.86      0.86       400


=== ROC AUC ===
0.9046794744459263

=== Top Features ===
Hourly Transaction Count        0.270046
Chip Usage Count                0.227688
MCC Diversity                   0.149880
Zip Code Count                  0.132599
City/State Zip Diversity        0.120867
Mean Hourly Spending            0.068430
Hourly Merchant Category Max    0.030490
Hourly Error Rate               0.000000
Transaction Frequency           0.000000
dtype: float64


In [ ]:
report2 = classification_report(y_test, y_pred)
roc2 = roc_auc_score(y_test, y_prob)

In [ ]:
# precision    recall  f1-score   support

#           No       0.87      0.72      0.79       131
#          Yes       0.87      0.95      0.91       269

#     accuracy                           0.87       400
#    macro avg       0.87      0.83      0.85       400
# weighted avg       0.87      0.87      0.87       400

# ROC-AUC: 0.9174210391895343

In [ ]:
print(report1)
print(report2)


              precision    recall  f1-score   support

           0       0.88      0.71      0.78       131
           1       0.87      0.95      0.91       269

    accuracy                           0.87       400
   macro avg       0.87      0.83      0.85       400
weighted avg       0.87      0.87      0.87       400

              precision    recall  f1-score   support

           0       0.87      0.69      0.77       131
           1       0.86      0.95      0.90       269

    accuracy                           0.86       400
   macro avg       0.87      0.82      0.84       400
weighted avg       0.87      0.86      0.86       400



In [ ]:
print(roc1)
print(roc2)

0.9080564147677289
0.9046794744459263
